Questo è il Notebook relativo alla task 4 del progetto.

In [10]:
#Librerie
import numpy as np
from scipy.sparse import csc_matrix, coo_matrix
from scipy.sparse.linalg import spsolve_triangular
from sksparse.cholmod import cholesky

#funzione che legge le matrici in formato COO e restituisce la matrice in formato CSC
def read_coo_to_csc(a_filename):
    # Caricamento di A e conversione in CSC
    data_A = np.loadtxt(a_filename)
    rows = data_A[:, 0].astype(int)
    cols = data_A[:, 1].astype(int)
    vals = data_A[:, 2]
    A_coo = coo_matrix((vals, (rows, cols)))
    A_csc = A_coo.tocsc()
    return A_csc

A_orig_CSC = read_coo_to_csc("A_orig.txt")
A_reordered_CSC = read_coo_to_csc("A_reordered.txt")

# Funzione che calcola la fattorizzazione di Cholesky di una matrice simmetrica definita positiva
def my_cholesky(A):
    factor = cholesky(A, order="natural")
    return csc_matrix(factor[0])

L_orig = my_cholesky(A_orig_CSC)
L_reordered = my_cholesky(A_reordered_CSC)

#stampo L per vedere se fin qui ho fatto tutto correttamente: se A è negativa, allora il codice mi darà errore
#print("Fattorizzazione di Cholesky della matrice originale (L):")
#print(L_orig)
#print("Fattorizzazione di Cholesky della matrice riordinata (L):")
#print(L_reordered)

#funzione che legge il termine noto b
def read_rhs(rhs_filename):
    data = np.loadtxt(rhs_filename)
    return data[:, 1] if data.ndim > 1 else data

b_orig = read_rhs("rhs_orig.txt")
b_reordered = read_rhs("rhs_reordered.txt")

#funzione che risolve il sistema lineare Ax = b usando la fattorizzazione di Cholesky
def solve_cholesky(L, b):
    # Risolvo Ly = b
    y = spsolve_triangular(L, b, lower=True)
    # Risolvo L^Tu = y
    u = spsolve_triangular(L.transpose(), y, lower=False)
    return u

# Calcolo delle soluzioni
u_orig = solve_cholesky(L_orig, b_orig)
u_reordered = solve_cholesky(L_reordered, b_reordered)

#  Calcolo dei residui relativi
res_orig = np.linalg.norm(A_orig_CSC.dot(u_orig) - b_orig) / np.linalg.norm(
    b_orig
)
res_reord = np.linalg.norm(
    A_reordered_CSC.dot(u_reordered) - b_reordered
) / np.linalg.norm(b_reordered)

# Stampa dei risultati di accuratezza ed efficienza
print("=== SISTEMA ORIGINALE ===")
print(f"Residuo relativo : {res_orig:.2e}")
print(f"Fill-in NNZ(L)   : {L_orig.nnz}\n")

print("=== SISTEMA RIORDINATO ===")
print(f"Residuo relativo : {res_reord:.2e}")
print(f"Fill-in NNZ(L)   : {L_reordered.nnz}\n")

print("Primi 10 elementi della soluzione:", u_orig[:10])
print("Primi 10 elementi della soluzione riordinata:", u_reordered[:10])


=== SISTEMA ORIGINALE ===
Residuo relativo : 5.58e-01
Fill-in NNZ(L)   : 29

=== SISTEMA RIORDINATO ===
Residuo relativo : 5.77e-01
Fill-in NNZ(L)   : 26

Primi 10 elementi della soluzione: [4.47662500e-01 1.43329176e-17 0.00000000e+00 0.00000000e+00
 0.00000000e+00 0.00000000e+00 4.50604827e-19 0.00000000e+00
 0.00000000e+00]
Primi 10 elementi della soluzione riordinata: [4.47662500e-01 2.13642101e-02 0.00000000e+00 2.14585444e-02
 2.76885926e-04 0.00000000e+00 0.00000000e+00 7.94438448e-18
 0.00000000e+00]
